# 03 Check WLASL1000 Keypoint Output

## Purpose
This notebook validates extracted WLASL1000 keypoint files and creates the final clean training index.

## Main output

```text
data/processed/ASL/WLASL1000/wlasl1000_clean_keypoint_index.csv
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
KEYPOINT_DIR = BASE_DIR / "keypoints"

VIDEO_INDEX_FILE = BASE_DIR / f"{PREFIX}_video_index.csv"
KEYPOINT_INDEX_FILE = BASE_DIR / f"{PREFIX}_keypoint_index.csv"
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"

video_df = pd.read_csv(VIDEO_INDEX_FILE)

print("Videos in usable index:", len(video_df))
print("Classes:", video_df["label_id"].nunique())
print("Extracted .npy files:", len(list(KEYPOINT_DIR.glob("*.npy"))))

## 1. Rebuild keypoint index

In [ ]:
records = []

for _, row in video_df.iterrows():
    keypoint_path = KEYPOINT_DIR / f"{row['video_id']}.npy"

    if keypoint_path.exists():
        records.append({
            "video_id": row["video_id"],
            "gloss": row["gloss"],
            "label_id": int(row["label_id"]),
            "original_class_id": int(row["original_class_id"]),
            "video_path": row["video_path"],
            "keypoint_path": str(keypoint_path)
        })

keypoint_df = pd.DataFrame(records)
keypoint_df.to_csv(KEYPOINT_INDEX_FILE, index=False)

print("Saved keypoint index:", KEYPOINT_INDEX_FILE)
print("Keypoint samples:", len(keypoint_df))
print("Classes:", keypoint_df["label_id"].nunique())

keypoint_df.head()

## 2. Check missing files

In [ ]:
expected_ids = set(video_df["video_id"].astype(str))
actual_ids = set([p.stem for p in KEYPOINT_DIR.glob("*.npy")])

missing = expected_ids - actual_ids

print("Missing keypoint files:", len(missing))
list(missing)[:20]

## 3. Check shape and zero ratio

In [ ]:
shape_records = []

for _, row in keypoint_df.iterrows():
    arr = np.load(row["keypoint_path"])

    shape_records.append({
        "video_id": row["video_id"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "shape": arr.shape,
        "num_frames": arr.shape[0],
        "num_features": arr.shape[1] if len(arr.shape) == 2 else None,
        "zero_ratio": float(np.mean(arr == 0))
    })

shape_df = pd.DataFrame(shape_records)

print("Unique shapes:")
print(shape_df["shape"].value_counts())

bad_shapes = shape_df[(shape_df["num_frames"] != 60) | (shape_df["num_features"] != 258)]

print("Bad shape files:", len(bad_shapes))
print("\nZero ratio summary:")
print(shape_df["zero_ratio"].describe())

shape_df.sort_values("zero_ratio", ascending=False).head(20)

## 4. Create clean index

In [ ]:
valid_ids = set(
    shape_df[
        (shape_df["num_frames"] == 60) &
        (shape_df["num_features"] == 258) &
        (shape_df["zero_ratio"] < 0.95)
    ]["video_id"].astype(str)
)

clean_df = keypoint_df[keypoint_df["video_id"].astype(str).isin(valid_ids)].copy()

zero_lookup = shape_df.set_index(shape_df["video_id"].astype(str))["zero_ratio"].to_dict()
clean_df["zero_ratio"] = clean_df["video_id"].astype(str).map(zero_lookup)

clean_df.to_csv(CLEAN_INDEX_FILE, index=False)

print("Saved clean index:", CLEAN_INDEX_FILE)
print("Clean samples:", len(clean_df))
print("Clean classes:", clean_df["label_id"].nunique())

clean_df.head()

## 5. Class distribution after cleaning

In [ ]:
class_counts = clean_df["gloss"].value_counts()

print("Class distribution after cleaning")
print("---------------------------------")
print("Classes:", len(class_counts))
print("Minimum samples per class:", class_counts.min())
print("Maximum samples per class:", class_counts.max())
print("Average samples per class:", round(class_counts.mean(), 2))

class_counts.head(30)

In [ ]:
plt.figure(figsize=(20, 5))
class_counts.plot(kind="bar")
plt.title("WLASL1000 Clean Keypoint Samples per Class")
plt.xlabel("Gloss")
plt.ylabel("Number of Samples")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()